# 05 - Adapter Comparison

Compare RSA vs ECDSA vs Kyber vs Hybrid adapters with summary tables and charts.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams.update({"font.family": "serif", "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

try:
    %store -r df
    print(f"Loaded {len(df):,} events")
except:
    np.random.seed(42)
    n = 40000
    algos = np.repeat(["RSA-2048", "ECDSA-P256", "Kyber-768", "RSA+Kyber-Hybrid"], n // 4)
    base_latency = {"RSA-2048": 1500, "ECDSA-P256": 150, "Kyber-768": 80, "RSA+Kyber-Hybrid": 200}
    df = pd.DataFrame({
        "algorithm": algos,
        "latency_us": [int(np.random.lognormal(np.log(base_latency[a]), 0.3)) for a in algos],
    })
    print(f"Created mock data with {len(df):,} events")


In [ ]:
# Summary statistics table
if "algorithm" in df.columns:
    def summary_stats(x):
        return pd.Series({
            "N": len(x),
            "Mean": x.mean(),
            "Std": x.std(),
            "p50": x.quantile(0.50),
            "p95": x.quantile(0.95),
            "p99": x.quantile(0.99),
        })
    
    summary = df.groupby("algorithm")["latency_us"].apply(summary_stats).unstack()
    print("Latency Statistics by Algorithm (μs):")
    print(summary.round(1).to_string())


In [ ]:
# Visual comparison
if "algorithm" in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    algorithms = df["algorithm"].unique()
    colors = plt.cm.Set2(np.linspace(0, 1, len(algorithms)))
    
    # CDF comparison
    ax1 = axes[0, 0]
    for algo, color in zip(algorithms, colors):
        data = df[df["algorithm"] == algo]["latency_us"].values
        sorted_data = np.sort(data)
        cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
        ax1.plot(sorted_data, cdf, linewidth=2, label=algo, color=color)
    ax1.set_xlabel("Latency (μs)")
    ax1.set_ylabel("CDF")
    ax1.set_title("Latency CDF by Algorithm")
    ax1.legend()
    
    # Box plot
    ax2 = axes[0, 1]
    data_by_algo = [df[df["algorithm"] == algo]["latency_us"].values for algo in algorithms]
    bp = ax2.boxplot(data_by_algo, labels=algorithms, patch_artist=True)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax2.set_ylabel("Latency (μs)")
    ax2.set_title("Latency Distribution by Algorithm")
    ax2.tick_params(axis='x', rotation=45)
    
    # Mean comparison with error bars
    ax3 = axes[1, 0]
    means = df.groupby("algorithm")["latency_us"].mean()
    stds = df.groupby("algorithm")["latency_us"].std()
    x = range(len(means))
    ax3.bar(x, means.values, yerr=stds.values, capsize=5, color=colors, alpha=0.7)
    ax3.set_xticks(x)
    ax3.set_xticklabels(means.index, rotation=45)
    ax3.set_ylabel("Latency (μs)")
    ax3.set_title("Mean Latency ± Std Dev")
    
    # p99 comparison
    ax4 = axes[1, 1]
    p99 = df.groupby("algorithm")["latency_us"].quantile(0.99)
    ax4.bar(range(len(p99)), p99.values, color=colors, alpha=0.7)
    ax4.set_xticks(range(len(p99)))
    ax4.set_xticklabels(p99.index, rotation=45)
    ax4.set_ylabel("p99 Latency (μs)")
    ax4.set_title("99th Percentile Latency")
    
    plt.tight_layout()
    plt.show()


# 05 - Adapter Comparison

Compare cryptographic adapters: RSA, ECDSA, PQC (Kyber/Dilithium), and Hybrid modes.

## Objectives
- Compare latency across adapters
- Generate summary tables
- Create comparison charts
- Identify performance trade-offs


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load multiple experiments for comparison
# Modify paths as needed
experiments = {
    "RSA-2048": "../data/exp_rsa/merged/merged.parquet",
    "ECDSA-P256": "../data/exp_ecdsa/merged/merged.parquet",
    "ML-KEM-768": "../data/exp_kyber/merged/merged.parquet",
    "Hybrid": "../data/exp_hybrid/merged/merged.parquet",
}

dfs = {}
for name, path in experiments.items():
    try:
        dfs[name] = pd.read_parquet(path)
        dfs[name]["adapter"] = name
        print(f"✓ Loaded {name}: {len(dfs[name]):,} records")
    except FileNotFoundError:
        print(f"✗ {name}: file not found")


In [ ]:
# Combine and compare
if dfs:
    combined = pd.concat(dfs.values(), ignore_index=True)
    
    # Summary table
    summary = combined.groupby("adapter")["latency_us"].agg([
        "count", "mean", "std", 
        lambda x: x.quantile(0.50),
        lambda x: x.quantile(0.99)
    ])
    summary.columns = ["Count", "Mean (μs)", "Std", "p50 (μs)", "p99 (μs)"]
    print("\n=== Adapter Comparison ===")
    display(summary.round(2))
    
    # Box plot comparison
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(data=combined, x="adapter", y="latency_us", ax=ax, palette="Set2")
    ax.set_xlabel("Adapter")
    ax.set_ylabel("Latency (μs)")
    ax.set_title("Latency Comparison by Adapter")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
